# Definition of Wigner function

Here, we calculate the moments of the Wigner function using SymPy:

$$S_{ft}=\int \mathrm{d}k\: \tilde{\xi}(f+1/2k)\tilde{\xi}^{\ast}(f-1/2k) e^{2\pi i kt}.$$

The formula we had found analytically is

$$\langle S_{t_1 f_1} S_{t_2 f_2}...S_{t_n f_n} \rangle = \int \mathrm{d}\vec{k}\: e^{2\pi i \vec{k}\vec{t}} \bigg\langle \prod_{i=1}^{2n} \tilde{\xi}(a_i)\bigg\rangle=\int \mathrm{d}\vec{k}\: e^{2\pi i \vec{k}\vec{t}} \sum_{p\in \mathbb{P}} \prod_{(i,j)= p} \Xi_{a(i),a(j)}=\int \mathrm{d}\vec{k}\: e^{2\pi i \vec{k}\vec{t}} \sum_{p\in \mathbb{P}} \prod_{(i,j)= p} \delta(a(i)-a(j)),$$

where

$$ a_i = \frac{1}{2} k_{r(i)} + (-1)^{i+1} f_{r(i)}$$

and

$$r(i):=\mathrm{ceil}(i/2).$$

In [11]:
from itertools import combinations

import numpy as np
import sympy as sym
from IPython.display import display, Math

inf_bounds = (-sym.oo, sym.oo)

def latex_this(expr, apply_indent=False):
    if apply_indent:
        indent = "\\hspace{3em}"
    else:
        indent = ""
    display(Math(indent + sym.latex(expr)))

In [11]:
from sympy import symbols, exp, pi, I, DiracDelta, oo, Integral
from sympy import expand, collect

def identify_delta_integral(integral_expr):
    """
    Identify integrals of the form ∫ e^(2πi*k*f(t)) dk from -∞ to ∞
    and replace them with DiracDelta(f(t))/(2π) (with appropriate normalization)

    Returns the simplified expression with Dirac deltas substituted.
    """

    # If it's not an Integral, return as-is
    if not isinstance(integral_expr, Integral):
        return integral_expr

    # Extract integration variable and limits
    integrand = integral_expr.args[0]
    integration_vars = integral_expr.args[1:]

    # We're looking for single integrals from -oo to oo
    if len(integration_vars) != 1:
        # More than one integration variable detected, returning integral expression without inserting dirac delta
        return integral_expr

    int_var, lower, upper = integration_vars[0]

    # Check if limits are -oo to oo
    if lower != -oo or upper != oo:
        # Limits are not -inf and +inf, returning integral expression without inserting dirac delta
        return integral_expr

    # Expand and simplify the integrand
    integrand = expand(integrand)

    # Try to match pattern: exp(A*I*int_var) where A doesn't contain int_var
    # This handles cases like exp(2*pi*I*k*t) or exp(2*pi*I*k*(t1+t2))

    # Collect exponentials
    if integrand.is_Mul:
        # Product of exponentials
        exp_args = []
        other_factors = []

        for factor in integrand.args:
            if isinstance(factor, exp):
                exp_args.append(factor.args[0])
            else:
                other_factors.append(factor)

        if exp_args:
            # Combine exponents
            total_exponent = sum(exp_args)
        else:
            return integral_expr

        other_part = sym.Mul(*other_factors) if other_factors else sym.S.One
    elif isinstance(integrand, exp):
        total_exponent = integrand.args[0]
        other_part = sym.S.One
    else:
        return integral_expr

    # Now check if total_exponent has the form: coefficient*1j*int_var*something
    # where 'something' doesn't depend on int_var

    # Expand the exponent
    total_exponent = expand(total_exponent)

    # Collect terms with int_var
    collected = collect(total_exponent, int_var, evaluate=False)

    # The coefficient of int_var should be I*2*pi*f(other_vars)
    if int_var in collected:
        coeff = collected[int_var]
        constant_term = collected.get(sym.S.One, sym.S.Zero)

        # Check if coefficient has the form (2 pi i) * something
        # Factor out I
        coeff_no_i = coeff / (I*sym.pi*2)

        # Check if coeff_no_i is independent of int_var
        if int_var not in coeff_no_i.free_symbols:
            # We have ∫ exp(2pi I * int_var * coeff_no_i) * exp(constant_term) * other_part d(int_var)
            # This equals: exp(constant_term) * other_part * ∫ exp(2pi I * int_var * coeff_no_i) d(int_var)
            # = exp(constant_term) * other_part * δ(coeff_no_i)

            # Standard normalization: ∫ e^(i*k*x) dk = 2π δ(x)
            result = DiracDelta(coeff_no_i) * exp(constant_term) * other_part
            return result

    return integral_expr


def identify_deltas_recursively(expr):
    """
    Recursively apply delta integral identification to an expression
    """
    if isinstance(expr, Integral):
        return identify_delta_integral(expr)
    elif expr.args:
        new_args = [identify_deltas_recursively(arg) for arg in expr.args]
        return expr.func(*new_args)
    else:
        return expr


def identify_and_simplify_deltas_recursively(expr, simplify_num=2):
    res = identify_deltas_recursively(expr)
    for _ in range(simplify_num):
        res = sym.simplify(res)
    return res

unit_test_examples = False
if unit_test_examples:
    # ============ EXAMPLES ============
    # Assuming you have a function latex_this(expression) defined

    print("Example 1: Your specific integral")
    print("=" * 70)
    k_2, t_1, t_2 = symbols('k_2 t_1 t_2', real=True)

    integral1 = Integral(exp(2*I*pi*k_2*t_1)*exp(2*I*pi*k_2*t_2), (k_2, -oo, oo))
    print(f"Original:")
    latex_this(integral1)

    result1 = identify_and_simplify_deltas_recursively(integral1)
    print(f"\nSimplified:")
    latex_this(result1)

    print("\n" + "=" * 70)
    print("Example 2: Single variable case")
    print("=" * 70)
    k, t = symbols('k t', real=True)

    integral2 = Integral(exp(2*I*pi*k*t), (k, -oo, oo))
    print(f"Original:")
    latex_this(integral2)

    result2 = identify_and_simplify_deltas_recursively(integral2)
    print(f"\nSimplified:")
    latex_this(result2)

    print("\n" + "=" * 70)
    print("Example 3: With additional factor")
    print("=" * 70)
    a = symbols('a', real=True)

    integral3 = Integral(a * exp(2*I*pi*k*t), (k, -oo, oo))
    print(f"Original:")
    latex_this(integral3)

    result3 = identify_and_simplify_deltas_recursively(integral3)
    print(f"\nSimplified:")
    latex_this(result3)

    print("\n" + "=" * 70)
    print("Example 4: Complex expression with multiple terms")
    print("=" * 70)

    # This would appear in expressions like exp(2πik(t1+t2-t3))
    integral4 = Integral(exp(2*I*pi*k_2*(t_1 + t_2)), (k_2, -oo, oo))
    print(f"Original:")
    latex_this(integral4)

    result4 = identify_and_simplify_deltas_recursively(integral4)
    print(f"\nSimplified:")
    latex_this(result4)

    print("\n" + "=" * 70)
    print("Example 5: Nested in larger expression")
    print("=" * 70)

    expr5 = 5 + integral1 * integral2
    print(f"Original:")
    latex_this(expr5)

    result5 = identify_and_simplify_deltas_recursively(expr5)
    print(f"\nSimplified:")
    latex_this(result5)

    # You can further simplify using the sifting property of delta
    print(f"\nNote: DiracDelta(t_1 + t_2) = δ(t_1 + t_2)")
    print(f"      This is zero unless t_1 + t_2 = 0")



In [12]:
def r_i(idx: int):
    """
    Resets the index, i.e. implements the following map:

    1  -> 1
    2  -> 1
    3  -> 2
    4  ->

    :param idx: An integer
    :return:
    """
    return int(np.ceil(idx/2))

def a_i_coefficients(order_of_moment):
    """
    For example:

        for i in range(N):
        coeff = sym.symbols(f"a_{i+1}")
        res.append(coeff)

        [latex_this(a) for a in a_i_coefficients(2)]

        >> a_1, a_2, a_3, a_4 (displayed with latex)

    :param order_of_moment:     The order of the desired Wigner function moment to be calculated
    :return:
    """
    res = []
    N = 2 * order_of_moment
    for i in range(N):
        ith_a_coeff = sym.symbols(f"a_{i+1}")
        res.append(ith_a_coeff)
    return res


def express_a_i_via_frequencies(a_i: sym.Symbol):
    """

    Takes an a_i symbol and spits out the right hand side of the definition of a_i:

        a_i -> 1/2 k_{r(i)} + (-1)^{i+1} f_{r(i)}

    :param a_i:     A sympy symbol that carries an index that this function can grab, in order to insert the definition of a_i via frequencies.
    :return:
    """
    i = int(a_i.__str__().split("_")[1])
    r = r_i(i)
    k = sym.symbols(f"k_{r}")
    f = sym.symbols(f"f_{r}")
    return 1/2 * k + (-1)**(i+1) * f


def calculate_pair_permutations(order_of_moment):
    """
    DEPRECATED
    Lifted from chatgpt
    For example:

    a,b,c,d = sym.symbols('a b c d')
    S = {a,b,c,d}

    out = set()

    for P in combinations(S, 2):
        P = frozenset(P)
        Q = frozenset(S - P)
        out.add(frozenset([P, Q]))   # unordered pair of unordered pairs

    result = [tuple(map(tuple, x)) for x in out]
    result

    >> [((a, d), (b, c)), ((b, d), (a, c)), ((d, c), (a, b))]


    :param order_of_moment:     The order of the desired Wigner function moment to be calculated
    :return:
    """
    my_a_i_coeffs = a_i_coefficients(order_of_moment=order_of_moment)

    S = set(my_a_i_coeffs)
    out = set()
    for P in combinations(S, 2):
        P = frozenset(P)
        Q = frozenset(S - P)
        out.add(frozenset([P, Q]))   # unordered pair of unordered pairs
    result = [tuple(map(tuple, x)) for x in out]

    return result

def calculate_pair_permutations_IMPROVED(order_of_moment):
    """
    Lifted from chatgpt.

    Example: order_of_moment=2 outputs:

        [((a_3, a_2), (a_1, a_4)), ((a_3, a_1), (a_2, a_4)), ((a_2, a_1), (a_3, a_4))] (3 summands out of 2 factors)

    Example: order_of_moment=3 outputs:

        [((a_2, a_4), (a_5, a_3), (a_6, a_1)), ((a_3, a_1), (a_5, a_4), (a_6, a_2)), ((a_6, a_4), (a_5, a_3), (a_2, a_1)), ((a_2, a_4), (a_5, a_1), (a_6, a_3)), ((a_6, a_5), (a_2, a_1), (a_3, a_4)), ((a_6, a_5), (a_3, a_2), (a_1, a_4)), ((a_3, a_1), (a_2, a_4), (a_6, a_5)), ((a_6, a_4), (a_3, a_2), (a_5, a_1)), ((a_6, a_3), (a_1, a_4), (a_5, a_2)), ((a_5, a_3), (a_6, a_2), (a_1, a_4)), ((a_5, a_4), (a_3, a_2), (a_6, a_1)), ((a_5, a_2), (a_3, a_4), (a_6, a_1)), ((a_2, a_1), (a_5, a_4), (a_6, a_3)), ((a_3, a_1), (a_6, a_4), (a_5, a_2)), ((a_5, a_1), (a_6, a_2), (a_3, a_4))]
        (15 summands out of 3 factors)
    """
    # Generate symbols
    my_a_i_coeffs = a_i_coefficients(order_of_moment=order_of_moment)
    S = list(my_a_i_coeffs)

    def recursive_pairs(elements):
        if not elements:
            return [()]
        all_pairings = []
        first = elements[0]
        for second in elements[1:]:
            pair = (first, second)
            remaining = [e for e in elements if e != first and e != second]
            for rest in recursive_pairs(remaining):
                all_pairings.append((pair,) + rest)
        return all_pairings

    # To remove duplicates due to unordered pairs, we can convert each pair to frozenset
    unique_pairings = set()
    for p in recursive_pairs(S):
        # Convert inner pairs to frozensets and outer tuple to frozenset
        outer = frozenset(frozenset(x) for x in p)
        unique_pairings.add(outer)

    # Convert back to tuple of tuples
    result = [tuple(tuple(x) for x in outer) for outer in unique_pairings]
    return result


def wicks_theorem(list_of_all_permutations: list, order_of_moment):
    to_sum = []
    for p in list_of_all_permutations:
        fac = sym.Mul(*[DiracDelta(a_ij[0] + a_ij[1]) for a_ij in p])
        to_sum.append(fac)
    return sum(to_sum)


def express_a_i_permutations_via_frequencies(a_i_permutations: list):
    """
    Takes a list like [[(a_1, a_2), (a_3, a_4)], [(a_1, a_3), (a_2, a_4)], etc...]
    and inserts for each element the output of the function express_a_i_via_frequencies.

    For example:

    turning a_i=a_1 to -->  f_1 + 0.5*k_1
    turning a_i=a_3 to -->  f_2 + 0.5*k_2
    turning a_i=a_2 to -->  -f_1 + 0.5*k_1
    turning a_i=a_4 to -->  -f_2 + 0.5*k_2

    turning a_i=a_1 to -->  f_1 + 0.5*k_1
    turning a_i=a_2 to -->  -f_1 + 0.5*k_1
    turning a_i=a_3 to -->  f_2 + 0.5*k_2
    turning a_i=a_4 to -->  -f_2 + 0.5*k_2

    turning a_i=a_1 to -->  f_1 + 0.5*k_1
    turning a_i=a_4 to -->  -f_2 + 0.5*k_2
    turning a_i=a_2 to -->  -f_1 + 0.5*k_1
    turning a_i=a_3 to -->  f_2 + 0.5*k_2


    :param a_i_permutations:    A list of pair orderings of a_i.
    :return:
    """
    l = []
    for p in a_i_permutations:
        transformed_p = list(
            tuple(express_a_i_via_frequencies(a_i) for a_i in tpl) for tpl in p
        )
        l.append(transformed_p)
    return l


def get_correlation_structure(order_of_moment:int):
    """
    For example:

        get_integrand(order_of_moment=2)

        >> DiracDelta(2*f_1)*DiracDelta(2*f_2) + DiracDelta(-f_1 - f_2 + 0.5*k_1 - 0.5*k_2)*DiracDelta(f_1 + f_2 + 0.5*k_1 - 0.5*k_2) + DiracDelta(-f_1 + f_2 + 0.5*k_1 - 0.5*k_2)*DiracDelta(f_1 - f_2 + 0.5*k_1 - 0.5*k_2)

    :param order_of_moment:     The order of the desired Wigner function moment to be calculated
    :return:
    """
    all_a_i_combinations = calculate_pair_permutations_IMPROVED(order_of_moment=order_of_moment)  # [[(a_1, a_2), (a_3, a_4)], [(a_1, a_3), (a_2, a_4)], etc...]
    all_frequency_combinations = express_a_i_permutations_via_frequencies(a_i_permutations=all_a_i_combinations)  # same but the definition via frequencies inserted

    integrand = wicks_theorem(order_of_moment=order_of_moment, list_of_all_permutations=all_frequency_combinations)
    return integrand


In [13]:
def get_integrand_and_integration_variables(order_of_moment:int):
    """
    Builds e^{2pi \vec{k}\vec{t}} * correlation from wicks theorem and returns it along the integration variables (\vec{k})
    :param order_of_moment:
    :return:
    """
    print("...Getting correlation structure through Wick's theorem")
    corr = get_correlation_structure(order_of_moment=order_of_moment)
    print("\tDone")
    print("...Adding Fourier phase factors to build integrand")

    # Define Fourier factor. First define the new t_vector
    t_vector = []
    for i in range(order_of_moment):
        t_vector.append(sym.symbols(f"t_{i+1}"))

    # Then get all k_vector elements by extracting them from the integrand expression
    k_vector = [k_i for k_i in corr.free_symbols if "k" in k_i.__str__()]

    def index(sym):
        return int(sym.name.split('_')[1])

    # Ensure that k_1 gets mapped to t_1 and not to t_2, i.e. just ensure same ordering of the vectors
    k_map = {index(k): k for k in k_vector}
    t_map = {index(t): t for t in t_vector}

    variable_with_conjugate_pairs = [(k_map[i], t_map[i]) for i in k_map.keys() & t_map.keys()]
    conjugate_pair_products = [2*sym.pi * 1j*i*j for i, j in variable_with_conjugate_pairs]  #  builds pairs of i k t

    fourier_factor = sym.exp(sum(conjugate_pair_products))
    print("\tDone")
    return fourier_factor.factor()*corr, k_vector


def calculate_wigner_function_moment(order_of_moment:int, number_of_recursive_simplifications:int=2):
    if order_of_moment < 2:
        raise ValueError("order_of_moment must be greater than 1")
    integrand, k_vector = get_integrand_and_integration_variables(order_of_moment=order_of_moment)

    integrand_as_sum_type = integrand.expand()

    if not isinstance(integrand_as_sum_type, sym.core.add.Add):
        raise ValueError("integrand at this point needs to be expanded and of summation type, since I will be integrating each summand by its own. This should have worked.")

    all_summands = integrand_as_sum_type.args

    integrated_summands = []
    print(f"...Performing {len(k_vector)} integrals over {len(all_summands)} terms")
    for integral_candidate in all_summands:
        integral_expr_k_i = integral_candidate
        for k_i in k_vector:
            integral_expr_k_i =  sym.integrate(integral_expr_k_i, (k_i, *inf_bounds))
            integral_expr_k_i = identify_and_simplify_deltas_recursively(integral_expr_k_i, simplify_num=number_of_recursive_simplifications)

        integrated_summands.append(integral_expr_k_i)

    return sym.Add(*integrated_summands)

In [14]:
import sympy as sp
from sympy import symbols, exp, pi, I, DiracDelta, Add, Mul, simplify

def simplify_exp_under_dirac(expr):
    """
    Simplify exponentials in an expression based on Dirac delta constraints.

    For each summand:
    1. Extract all DiracDelta factors and exponentials
    2. Check if DiracDelta constraints make exponentials equal to 1
    3. Simplify accordingly
    """

    # If it's a sum, process each term separately
    if isinstance(expr, Add):
        simplified_terms = [simplify_exp_under_dirac(term) for term in expr.args]
        return Add(*simplified_terms)

    # For a single term (product or atomic expression)
    return _simplify_single_term(expr)


def _simplify_single_term(term):
    """
    Simplify a single term by checking if Dirac deltas constrain exponentials.
    """

    # If it's not a product, return as-is
    if not isinstance(term, Mul):
        return term

    # Separate the term into components
    deltas = []
    exponentials = []
    other_factors = []

    for factor in term.args:
        if isinstance(factor, DiracDelta):
            deltas.append(factor)
        elif isinstance(factor, exp):
            exponentials.append(factor)
        else:
            other_factors.append(factor)

    # If no deltas or no exponentials, nothing to simplify
    if not deltas or not exponentials:
        return term

    # Extract constraints from Dirac deltas
    # DiracDelta(expr) means expr = 0
    constraints = [delta.args[0] for delta in deltas]

    # Try to simplify each exponential using the constraints
    simplified_exps = []
    for exp_factor in exponentials:
        exponent = exp_factor.args[0]

        # Substitute each constraint and see if exponent becomes 0
        simplified_exp = exponent
        for constraint in constraints:
            # Try to solve constraint for variables and substitute
            # DiracDelta(a - b) means a = b, so we can substitute a -> b
            simplified_exp = _try_apply_constraint(simplified_exp, constraint)

        # If exponent simplifies to 0, exp becomes 1
        simplified_exp = simplify(simplified_exp)
        if simplified_exp == 0:
            simplified_exps.append(sp.S.One)
        else:
            simplified_exps.append(exp(simplified_exp))

    # Reconstruct the term
    result = Mul(*other_factors, *deltas, *simplified_exps)
    return result


def _try_apply_constraint(expr, constraint):
    """
    Try to apply a constraint (from DiracDelta) to simplify an expression.
    For DiracDelta(a + b), we know a = -b, so substitute accordingly.
    """

    # Get all variables in the constraint
    constraint_vars = constraint.free_symbols

    # Try to solve the constraint for each variable
    for var in constraint_vars:
        try:
            # Solve constraint = 0 for var
            solutions = sp.solve(constraint, var)
            if solutions:
                # Use the first solution to substitute
                expr = expr.subs(var, solutions[0])
                break
        except:
            continue

    return expr

In [15]:
import sympy as sp
from sympy import symbols, exp, pi, I, DiracDelta, Add, Mul, cos, simplify, expand

def combine_exponentials_in_term(term):
    """
    Combine all exponential factors in a term into a single exponential.
    e.g., exp(a)*exp(b)*exp(c) -> exp(a+b+c)
    """
    if not isinstance(term, Mul):
        return term

    exp_factors = []
    other_factors = []

    for factor in term.args:
        if isinstance(factor, exp):
            exp_factors.append(factor.args[0])  # Get the exponent
        else:
            other_factors.append(factor)

    if not exp_factors:
        return term

    # Combine all exponents
    combined_exponent = sum(exp_factors)
    combined_exponent = expand(combined_exponent)

    # Reconstruct term
    if other_factors:
        return Mul(*other_factors, exp(combined_exponent))
    else:
        return exp(combined_exponent)


def find_and_pair_conjugates(expr):
    """
    Find conjugate pairs in a sum and combine them into cosine terms.

    For terms A*exp(i*x) and A*exp(-i*x), replace with 2*A*cos(x)
    """

    # If not a sum, return as-is
    if not isinstance(expr, Add):
        return expr

    # First, combine exponentials in each term
    terms = [combine_exponentials_in_term(term) for term in expr.args]

    # Separate terms with and without exponentials
    exp_terms = []
    non_exp_terms = []

    for term in terms:
        has_exp = False
        if isinstance(term, Mul):
            has_exp = any(isinstance(f, exp) for f in term.args)
        elif isinstance(term, exp):
            has_exp = True

        if has_exp:
            exp_terms.append(term)
        else:
            non_exp_terms.append(term)

    # Try to find conjugate pairs
    paired_indices = set()
    cosine_terms = []

    for i, term1 in enumerate(exp_terms):
        if i in paired_indices:
            continue

        # Extract coefficient and exponential from term1
        coeff1, exp1 = extract_coeff_and_exp(term1)

        if exp1 is None:
            # No exponential found, keep as-is
            cosine_terms.append(term1)
            continue

        # Look for conjugate partner
        found_pair = False
        for j, term2 in enumerate(exp_terms[i+1:], start=i+1):
            if j in paired_indices:
                continue

            coeff2, exp2 = extract_coeff_and_exp(term2)

            if exp2 is None:
                continue

            # Check if exp1 and exp2 are conjugates
            # exp(ix) and exp(-ix) are conjugates if their exponents sum to 0
            exponent_sum = simplify(exp1 + exp2)

            if exponent_sum == 0 and simplify(coeff1 - coeff2) == 0:
                # Found a conjugate pair!
                # A*exp(ix) + A*exp(-ix) = 2*A*cos(x)
                # Need to extract the argument (remove the I)

                # exp1 should be I*something
                if exp1.has(I):
                    # Factor out I
                    real_part = simplify(exp1 / I)
                    cosine_term = 2 * coeff1 * cos(real_part)
                    cosine_terms.append(cosine_term)
                    paired_indices.add(i)
                    paired_indices.add(j)
                    found_pair = True
                    break

        if not found_pair:
            # No pair found, keep original term
            cosine_terms.append(term1)

    # Combine all terms
    result = Add(*non_exp_terms, *cosine_terms)
    return result


def extract_coeff_and_exp(term):
    """
    Extract coefficient and exponential exponent from a term.

    Returns (coefficient, exponent) where term = coefficient * exp(exponent)
    """
    if isinstance(term, exp):
        return sp.S.One, term.args[0]
    elif isinstance(term, Mul):
        exp_factor = None
        other_factors = []

        for factor in term.args:
            if isinstance(factor, exp):
                if exp_factor is not None:
                    # Multiple exponentials? Should have been combined
                    return None, None
                exp_factor = factor.args[0]
            else:
                other_factors.append(factor)

        if exp_factor is not None:
            coeff = Mul(*other_factors) if other_factors else sp.S.One
            return coeff, exp_factor

    return None, None


unit_test = False
if unit_test:
    # ============ EXAMPLES ============

    print("Example: Your third-order moment terms")
    print("=" * 80)

    # Define symbols
    f_1, f_2, f_3, t_1, t_2, t_3 = symbols('f_1 f_2 f_3 t_1 t_2 t_3', real=True)

    # Your expression (subset of terms for demonstration)
    expr = (
        1.0 +
        2*DiracDelta(2*f_1 - 2*f_2)*DiracDelta(t_1 - t_2) +
        2*DiracDelta(2*f_1 + 2*f_2)*DiracDelta(t_1 - t_2) +
        4*exp(-4*I*pi*f_1*t_2)*exp(4*I*pi*f_1*t_3)*exp(-4*I*pi*f_2*t_1)*exp(4*I*pi*f_2*t_3)*exp(-4*I*pi*f_3*t_1)*exp(4*I*pi*f_3*t_2) +
        4*exp(-4*I*pi*f_1*t_2)*exp(4*I*pi*f_1*t_3)*exp(-4*I*pi*f_2*t_1)*exp(4*I*pi*f_2*t_3)*exp(4*I*pi*f_3*t_1)*exp(-4*I*pi*f_3*t_2) +
        4*exp(-4*I*pi*f_1*t_2)*exp(4*I*pi*f_1*t_3)*exp(4*I*pi*f_2*t_1)*exp(-4*I*pi*f_2*t_3)*exp(-4*I*pi*f_3*t_1)*exp(4*I*pi*f_3*t_2) +
        4*exp(-4*I*pi*f_1*t_2)*exp(4*I*pi*f_1*t_3)*exp(4*I*pi*f_2*t_1)*exp(-4*I*pi*f_2*t_3)*exp(4*I*pi*f_3*t_1)*exp(-4*I*pi*f_3*t_2)
    )

    print("\nOriginal expression (showing first few terms):")
    for i, term in enumerate(expr.args[:7]):
        print(f"  Term {i+1}: {term}")

    print("\nStep 1: Combine exponentials in each term")
    print("-" * 80)
    combined_terms = [combine_exponentials_in_term(term) for term in expr.args]
    for i, term in enumerate(combined_terms[:7]):
        print(f"  Term {i+1}: {term}")

    print("\nStep 2: Find and pair conjugates")
    print("-" * 80)
    simplified = find_and_pair_conjugates(expr)
    print(f"\nSimplified result:")
    print(simplified)

    print("\n" + "=" * 80)
    print("Example 2: Simple conjugate pair")
    print("=" * 80)

    x = symbols('x', real=True)
    simple_expr = 3*exp(2*I*pi*x) + 3*exp(-2*I*pi*x) + 5

    print(f"\nOriginal: {simple_expr}")
    simplified_simple = find_and_pair_conjugates(simple_expr)
    print(f"Simplified: {simplified_simple}")
    print(f"Expected: 6*cos(2*pi*x) + 5")

    print("\n" + "=" * 80)
    print("Example 3: Multiple conjugate pairs")
    print("=" * 80)

    y = symbols('y', real=True)
    multi_expr = (
        2*exp(I*x) + 2*exp(-I*x) +
        4*exp(I*y) + 4*exp(-I*y) +
        1
    )

    print(f"\nOriginal: {multi_expr}")
    simplified_multi = find_and_pair_conjugates(multi_expr)
    print(f"Simplified: {simplified_multi}")
    print(f"Expected: 4*cos(x) + 8*cos(y) + 1")

In [16]:
import sympy as sp
from sympy import symbols, cos, pi, DiracDelta, Add, Mul, gcd, latex, simplify, factor

def simplify_delta_jacobians(expr):
    """
    Simplify Dirac deltas by extracting common factors (Jacobians).
    A*DiracDelta(B*x) -> A/|B| * DiracDelta(x)
    """
    if not isinstance(expr, Add):
        expr = Add(expr)

    new_terms = []
    for term in expr.args:
        new_terms.append(simplify_single_delta_term(term))

    return Add(*new_terms)


def simplify_single_delta_term(term):
    """
    Simplify a single term containing Dirac deltas.
    """
    if not isinstance(term, Mul):
        if isinstance(term, DiracDelta):
            simplified, jac = simplify_delta_with_jacobian(term)
            return simplified * jac
        return term

    # Separate deltas from other factors
    deltas = []
    other_factors = []

    for factor in term.args:
        if isinstance(factor, DiracDelta):
            deltas.append(factor)
        else:
            other_factors.append(factor)

    if not deltas:
        return term

    # Simplify each delta and collect Jacobians
    simplified_deltas = []
    jacobian = sp.S.One

    for delta in deltas:
        simplified, jac = simplify_delta_with_jacobian(delta)
        simplified_deltas.append(simplified)
        jacobian *= jac

    # Reconstruct term
    all_factors = other_factors + [jacobian] + simplified_deltas
    return Mul(*all_factors)


def simplify_delta_with_jacobian(delta):
    """
    Simplify DiracDelta(a*x) -> (1/|a|)*DiracDelta(x)
    Returns (simplified_delta, jacobian_factor)
    """
    arg = delta.args[0]

    # Check if argument is a simple multiple: a*x or a*x + b*y
    # For sums, find GCD of all coefficients
    if arg.is_Mul:
        coeff = arg.as_coeff_Mul()[0]
        rest = arg.as_coeff_Mul()[1]

        if coeff.is_number and coeff != 1:
            # DiracDelta(c*x) = (1/|c|)*DiracDelta(x)
            jacobian = 1 / sp.Abs(coeff)
            return DiracDelta(rest), jacobian

    elif arg.is_Add:
        # Find GCD of all coefficients
        coeffs = []
        for term_in_arg in arg.args:
            if term_in_arg.is_Mul:
                c = term_in_arg.as_coeff_Mul()[0]
                if c.is_number:
                    coeffs.append(abs(int(c)))
            elif term_in_arg.is_number:
                coeffs.append(abs(int(term_in_arg)))

        if coeffs:
            common_factor = sp.gcd(coeffs)
            if common_factor > 1:
                # Factor out the GCD
                new_arg = arg / common_factor
                jacobian = 1 / common_factor
                return DiracDelta(new_arg), jacobian

    return delta, sp.S.One


def to_latex_aligned(expr, num_of_deltas_before_linebreak=3, lhs_label="\\mathrm{moment}"):
    """
    Convert expression to LaTeX in align* environment with special formatting:
    - Scalars and delta terms grouped together (with line breaks every N deltas)
    - Each cosine term on its own line

    Args:
        expr: SymPy expression
        num_of_deltas_before_linebreak: Number of delta terms before inserting line break
        lhs_label: Label for left-hand side

    Returns:
        String containing LaTeX code
    """
    # Get all summands
    if isinstance(expr, Add):
        terms = expr.args
    else:
        terms = [expr]

    # Separate terms into categories
    scalar_terms = []
    delta_terms = []
    cosine_terms = []

    for term in terms:
        has_cos = term.has(cos)
        has_delta = term.has(DiracDelta)

        if has_cos:
            cosine_terms.append(term)
        elif has_delta:
            delta_terms.append(term)
        else:
            # Pure scalar
            scalar_terms.append(term)

    # Build the align* environment
    lines = []
    lines.append("\\begin{align*}")

    # First line: LHS = scalars + first batch of deltas
    first_line_parts = []

    # Add scalars first
    for scalar in scalar_terms:
        first_line_parts.append(latex(scalar))

    # Add deltas up to the limit
    delta_count = 0
    for delta_term in delta_terms[:num_of_deltas_before_linebreak]:
        first_line_parts.append(latex(delta_term))
        delta_count += 1

    # Build first line
    if first_line_parts:
        first_line = f"{lhs_label} &= "
        for i, part in enumerate(first_line_parts):
            if i == 0:
                first_line += part
            else:
                if part.startswith('-'):
                    first_line += ' ' + part
                else:
                    first_line += ' + ' + part
    else:
        first_line = f"{lhs_label} &= "

    # Check if we need continuation
    has_more = (delta_count < len(delta_terms)) or len(cosine_terms) > 0
    if has_more:
        first_line += " \\\\"

    lines.append(first_line)

    # Remaining delta terms, grouped by num_of_deltas_before_linebreak
    remaining_deltas = delta_terms[num_of_deltas_before_linebreak:]
    i = 0
    while i < len(remaining_deltas):
        line_parts = []
        for j in range(num_of_deltas_before_linebreak):
            if i + j >= len(remaining_deltas):
                break
            line_parts.append(latex(remaining_deltas[i + j]))

        # Build line
        line = "&\\quad "
        for k, part in enumerate(line_parts):
            if k == 0:
                if part.startswith('-'):
                    line += part
                else:
                    line += '+ ' + part
            else:
                if part.startswith('-'):
                    line += ' ' + part
                else:
                    line += ' + ' + part

        i += num_of_deltas_before_linebreak

        # Check if we need continuation
        has_more = (i < len(remaining_deltas)) or len(cosine_terms) > 0
        if has_more:
            line += " \\\\"

        lines.append(line)

    # Cosine terms - each on its own line
    for i, cos_term in enumerate(cosine_terms):
        cos_latex = latex(cos_term)

        # Add + or - appropriately
        if cos_latex.startswith('-'):
            line = f"&\\quad {cos_latex}"
        else:
            line = f"&\\quad + {cos_latex}"

        # Add \\ if not the last term
        if i < len(cosine_terms) - 1:
            line += " \\\\"

        lines.append(line)

    lines.append("\\end{align*}")

    return '\n'.join(lines)


# ============ EXAMPLE ============

unit_test=False
if unit_test:
    print("=" * 80)
    print("Processing your third-order moment expression")
    print("=" * 80)

    # Define symbols
    f_1, f_2, f_3, t_1, t_2, t_3 = symbols('f_1 f_2 f_3 t_1 t_2 t_3', real=True)

    # Your expression
    expr = (
        8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 - 4*f_2*t_1 + 4*f_2*t_3 - 4*f_3*t_1 + 4*f_3*t_2)) +
        8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 - 4*f_2*t_1 + 4*f_2*t_3 + 4*f_3*t_1 - 4*f_3*t_2)) +
        8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 + 4*f_2*t_1 - 4*f_2*t_3 - 4*f_3*t_1 + 4*f_3*t_2)) +
        8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 + 4*f_2*t_1 - 4*f_2*t_3 + 4*f_3*t_1 - 4*f_3*t_2)) +
        2*DiracDelta(2*f_1 - 2*f_2)*DiracDelta(t_1 - t_2) +
        2*DiracDelta(2*f_1 + 2*f_2)*DiracDelta(t_1 - t_2) +
        2*DiracDelta(2*f_1 - 2*f_3)*DiracDelta(t_1 - t_3) +
        2*DiracDelta(2*f_1 + 2*f_3)*DiracDelta(t_1 - t_3) +
        2*DiracDelta(2*f_2 - 2*f_3)*DiracDelta(t_2 - t_3) +
        2*DiracDelta(2*f_2 + 2*f_3)*DiracDelta(t_2 - t_3) +
        1.0
    )

    print("\nStep 2: Simplify Dirac delta Jacobians")
    print("-" * 80)
    simplified_deltas = simplify_delta_jacobians(expr)
    print(simplified_deltas)

    print("\nStep 3: Convert to LaTeX (3 terms per line)")
    print("-" * 80)
    latex_output = to_latex_aligned(simplified_deltas, num_of_deltas_before_linebreak=4)
    print(latex_output)

    print("\n" + "=" * 80)
    print("Copy the LaTeX code above for your thesis!")
    print("=" * 80)

In [17]:


def get_moment_and_simplify(order_of_moment, number_of_recursive_simplifications=2, apply_dirac_delta_constraints=False, collect_conjugate_exponential_terms=False,
                            display_intermediate_steps=True):
    """

    :param order_of_moment:                         The order of moment of the Wigner function to calculate
    :param number_of_recursive_simplifications:     How many intermediate recursive simplifications to apply
    :param apply_dirac_delta_constraints:           The final result might be 2e^{4pi t_1 (f1-f2)} δ(2f1-2f2). If this flag is True, this simplifies to δ(f1-f2).
    :param collect_conjugate_exponential_terms:     The final result may contain terms that are not coupled to a Dirac Delta, but rather look like sum of products of
                                                    exponentials. If this flag is True, it checks whether there are conjugate exponential terms and simplifies to
                                                    real cosine terms. In general, we should not expect for the final moment to be complex and therefore expect conjugate
                                                    terms. Only runs if apply_dirac_delta_constraints is set to True.
    :param display_intermediate_steps:              Whether to display and print intermediate steps
    :return:
    """
    moment = calculate_wigner_function_moment(order_of_moment=order_of_moment, number_of_recursive_simplifications=number_of_recursive_simplifications)
    moment = simplify_delta_jacobians(moment)

    mode = None
    if not apply_dirac_delta_constraints:
        mode = 1
    elif not collect_conjugate_exponential_terms:
        mode = 2
    else:
        mode = 3

    if display_intermediate_steps:
        print("\n")
        print("Calculated moment ")
        print("---------------------")
        print("\t\t" + moment.__str__())
        latex_this(moment, apply_indent=True)

    if mode == 1:
        return simplify_delta_jacobians(moment)

    if apply_dirac_delta_constraints:
        moment_constrained = simplify_exp_under_dirac(moment)
        moment_constrained = simplify_delta_jacobians(moment_constrained)

        if display_intermediate_steps:
            print("\n\nApplying implicit dirac delta constraints ")
            print("---------------------")
            print("\t\t" + moment_constrained.__str__())
            latex_this(moment_constrained, apply_indent=True)
        if mode == 2:
            return moment_constrained

        if collect_conjugate_exponential_terms:
            moment_constrained_and_simplified = find_and_pair_conjugates(moment_constrained)
            moment_constrained_and_simplified = simplify_delta_jacobians(moment_constrained_and_simplified)

            if display_intermediate_steps:
                print("\n\nPairing together free conjugate exponentials ")
                print("----------------------")
                print("\t\t" + moment_constrained_and_simplified.__str__())
                latex_this(moment_constrained_and_simplified, apply_indent=True)
            if mode == 3:
                return moment_constrained_and_simplified




In [18]:
moment_2 = get_moment_and_simplify(order_of_moment=2, number_of_recursive_simplifications=2, apply_dirac_delta_constraints=True, collect_conjugate_exponential_terms=False, display_intermediate_steps=True)
print("\n copy-pasteable latex string:")
print(to_latex_aligned(moment_2, num_of_deltas_before_linebreak=4))

...Getting correlation structure through Wick's theorem
	Done
...Adding Fourier phase factors to build integrand
	Done
...Performing 2 integrals over 3 terms


Calculated moment 
---------------------
		exp(4*I*pi*t_2*(f_1 - f_2))*DiracDelta(f_1 - f_2)*DiracDelta(t_1 - t_2) + exp(4*I*pi*t_2*(f_1 + f_2))*DiracDelta(f_1 + f_2)*DiracDelta(t_1 - t_2) + 1.0


<IPython.core.display.Math object>



Applying implicit dirac delta constraints 
---------------------
		DiracDelta(f_1 - f_2)*DiracDelta(t_1 - t_2) + DiracDelta(f_1 + f_2)*DiracDelta(t_1 - t_2) + 1.0


<IPython.core.display.Math object>


 copy-pasteable latex string:
\begin{align*}
\mathrm{moment} &= 1.0 + \delta\left(f_{1} + f_{2}\right) \delta\left(t_{1} - t_{2}\right) + \delta\left(f_{1} - f_{2}\right) \delta\left(t_{1} - t_{2}\right)
\end{align*}


In [55]:
moment_3 = get_moment_and_simplify(order_of_moment=3, number_of_recursive_simplifications=2, apply_dirac_delta_constraints=True, collect_conjugate_exponential_terms=False, display_intermediate_steps=True)
print("\n copy-pasteable latex string:")
print(to_latex_aligned(moment_3, num_of_deltas_before_linebreak=4))

...Getting correlation structure through Wick's theorem
	Done
...Adding Fourier phase factors to build integrand
	Done
...Performing 3 integrals over 15 terms


Calculated moment 
---------------------
		4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 - f_2*t_2 - f_3*t_1 + f_3*t_2 - t_1*(f_1 + f_2) + t_2*(f_1 + f_2) + t_3*(f_1 + f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 - f_2*t_2 + f_3*t_1 - f_3*t_2 - t_1*(f_1 + f_2) + t_2*(f_1 + f_2) + t_3*(f_1 + f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f_2*t_2 - f_3*t_1 + f_3*t_2 - t_1*(f_1 - f_2) + t_2*(f_1 - f_2) + t_3*(f_1 - f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f_2*t_2 + f_3*t_1 - f_3*t_2 - t_1*(f_1 - f_2) + t_2*(f_1 - f_2) + t_3*(f_1 - f_2))) + exp(4*I*pi*t_1*(f_1 - f_3))*DiracDelta(f_1 - f_3)*DiracDelta(t_1 - t_3) + exp(4*I*pi*t_1*(f_1 + f_3))*DiracDelta(f_1 + f_3)*DiracDelta(t_1 - t_3) + exp(4*I*pi*t_2*(f_1 - f_2))*DiracDelta(f_1 - f_2)*DiracDelta(t_1 - t_2) + exp(4*I*pi*t_2*(f_1 + f_2))*DiracDelta(f_1 + f_2)*DiracDelta(t_1 - t_2) + exp(4*I*pi*t

<IPython.core.display.Math object>



Applying implicit dirac delta constraints 
---------------------
		4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 - f_2*t_2 - f_3*t_1 + f_3*t_2 - t_1*(f_1 + f_2) + t_2*(f_1 + f_2) + t_3*(f_1 + f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 - f_2*t_2 + f_3*t_1 - f_3*t_2 - t_1*(f_1 + f_2) + t_2*(f_1 + f_2) + t_3*(f_1 + f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f_2*t_2 - f_3*t_1 + f_3*t_2 - t_1*(f_1 - f_2) + t_2*(f_1 - f_2) + t_3*(f_1 - f_2))) + 4*exp(4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f_2*t_2 + f_3*t_1 - f_3*t_2 - t_1*(f_1 - f_2) + t_2*(f_1 - f_2) + t_3*(f_1 - f_2))) + DiracDelta(f_1 - f_2)*DiracDelta(t_1 - t_2) + DiracDelta(f_1 + f_2)*DiracDelta(t_1 - t_2) + DiracDelta(f_1 - f_3)*DiracDelta(t_1 - t_3) + DiracDelta(f_1 + f_3)*DiracDelta(t_1 - t_3) + DiracDelta(f_2 - f_3)*DiracDelta(t_2 - t_3) + DiracDelta(f_2 + f_3)*DiracDelta(t_2 - t_3) + 1.0 + 4*exp(-4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f_2*t_2 + f_3*t_1 - f_3*t_2 - t_1*(f_1 - f_2) + t_2*(f_1 - f_2) + t_3*(f_1 - f_2))) + 4*exp(-4*I*pi*(f_1*t_1 - 2*f_1*t_2 + f

<IPython.core.display.Math object>


 copy-pasteable latex string:
\begin{align*}
\mathrm{moment} &= 1.0 + 4 e^{- 4 i \pi \left(f_{1} t_{1} - 2 f_{1} t_{2} + f_{2} t_{2} + f_{3} t_{1} - f_{3} t_{2} - t_{1} \left(f_{1} - f_{2}\right) + t_{2} \left(f_{1} - f_{2}\right) + t_{3} \left(f_{1} - f_{2}\right)\right)} + 4 e^{- 4 i \pi \left(f_{1} t_{1} - 2 f_{1} t_{2} + f_{2} t_{2} - f_{3} t_{1} + f_{3} t_{2} - t_{1} \left(f_{1} - f_{2}\right) + t_{2} \left(f_{1} - f_{2}\right) + t_{3} \left(f_{1} - f_{2}\right)\right)} + 4 e^{- 4 i \pi \left(f_{1} t_{1} - 2 f_{1} t_{2} - f_{2} t_{2} + f_{3} t_{1} - f_{3} t_{2} - t_{1} \left(f_{1} + f_{2}\right) + t_{2} \left(f_{1} + f_{2}\right) + t_{3} \left(f_{1} + f_{2}\right)\right)} + 4 e^{- 4 i \pi \left(f_{1} t_{1} - 2 f_{1} t_{2} - f_{2} t_{2} - f_{3} t_{1} + f_{3} t_{2} - t_{1} \left(f_{1} + f_{2}\right) + t_{2} \left(f_{1} + f_{2}\right) + t_{3} \left(f_{1} + f_{2}\right)\right)} + 4 e^{4 i \pi \left(f_{1} t_{1} - 2 f_{1} t_{2} + f_{2} t_{2} + f_{3} t_{1} - f_{3} t_{2} - t_{1} \left(f_

In [56]:
moment_4 = get_moment_and_simplify(order_of_moment=4, number_of_recursive_simplifications=2, apply_dirac_delta_constraints=True, collect_conjugate_exponential_terms=True, display_intermediate_steps=True)

...Getting correlation structure through Wick's theorem
	Done
...Adding Fourier phase factors to build integrand
	Done
...Performing 4 integrals over 105 terms


Calculated moment 
---------------------
		exp(4*I*pi*(f_1*t_1 + f_2*t_2 - f_3*t_1 - f_4*t_2))*DiracDelta(f_1 - f_3)*DiracDelta(f_2 - f_4)*DiracDelta(t_1 - t_3)*DiracDelta(t_2 - t_4) + exp(4*I*pi*(f_1*t_1 + f_2*t_2 - f_3*t_1 + f_4*t_2))*DiracDelta(f_1 - f_3)*DiracDelta(f_2 + f_4)*DiracDelta(t_1 - t_3)*DiracDelta(t_2 - t_4) + exp(4*I*pi*(f_1*t_1 + f_2*t_2 + f_3*t_1 - f_4*t_2))*DiracDelta(f_1 + f_3)*DiracDelta(f_2 - f_4)*DiracDelta(t_1 - t_3)*DiracDelta(t_2 - t_4) + exp(4*I*pi*(f_1*t_1 + f_2*t_2 + f_3*t_1 + f_4*t_2))*DiracDelta(f_1 + f_3)*DiracDelta(f_2 + f_4)*DiracDelta(t_1 - t_3)*DiracDelta(t_2 - t_4) + exp(4*I*pi*(f_1*t_1 + f_2*t_2 - f_3*t_2 - f_4*t_1))*DiracDelta(f_1 - f_4)*DiracDelta(f_2 - f_3)*DiracDelta(t_1 - t_4)*DiracDelta(t_2 - t_3) + exp(4*I*pi*(f_1*t_1 + f_2*t_2 - f_3*t_2 + f_4*t_1))*DiracDelta(f_1 + f_4)*DiracDelta(

<IPython.core.display.Math object>



Applying implicit dirac delta constraints 
---------------------
		4*exp(4*I*pi*(f_1*t_2 - f_1*t_4 - f_2*t_1 + f_2*t_4 - f_4*t_1 + f_4*t_2)) + 4*exp(4*I*pi*(f_1*t_2 - f_1*t_4 - f_2*t_1 + f_2*t_4 + f_4*t_1 - f_4*t_2)) + 4*exp(4*I*pi*(f_1*t_2 - f_1*t_4 + f_2*t_1 - f_2*t_4 - f_4*t_1 + f_4*t_2)) + 4*exp(4*I*pi*(f_1*t_2 - f_1*t_4 + f_2*t_1 - f_2*t_4 + f_4*t_1 - f_4*t_2)) + 4*exp(4*I*pi*(-f_2*t_1 + f_2*t_2 - f_3*t_2 + f_3*t_4 - f_4*t_1 + f_4*t_4))*DiracDelta(f_1 - f_2 + f_3 + f_4)*DiracDelta(t_1 - t_2 + t_3 - t_4) + 4*exp(4*I*pi*(-f_2*t_1 + f_2*t_2 - f_3*t_2 + f_3*t_4 - f_4*t_1 + f_4*t_4))*DiracDelta(f_1 + f_2 - f_3 - f_4)*DiracDelta(t_1 - t_2 + t_3 - t_4) + 4*exp(4*I*pi*(-f_2*t_1 + f_2*t_2 - f_3*t_2 + f_3*t_4 + f_4*t_1 - f_4*t_4))*DiracDelta(f_1 - f_2 + f_3 - f_4)*DiracDelta(t_1 - t_2 + t_3 - t_4) + 4*exp(4*I*pi*(-f_2*t_1 + f_2*t_2 - f_3*t_2 + f_3*t_4 + f_4*t_1 - f_4*t_4))*DiracDelta(f_1 + f_2 - f_3 + f_4)*DiracDelta(t_1 - t_2 + t_3 - t_4) + 4*exp(4*I*pi*(-f_2*t_1 + f_2*t_2 + f_3*t_2 - f_

<IPython.core.display.Math object>



Pairing together free conjugate exponentials 
----------------------
		8*cos(pi*(-4*f_1*t_2 + 4*f_1*t_4 - 4*f_2*t_1 + 4*f_2*t_4 - 4*f_4*t_1 + 4*f_4*t_2)) + 8*cos(pi*(-4*f_1*t_2 + 4*f_1*t_4 - 4*f_2*t_1 + 4*f_2*t_4 + 4*f_4*t_1 - 4*f_4*t_2)) + 8*cos(pi*(-4*f_1*t_2 + 4*f_1*t_4 + 4*f_2*t_1 - 4*f_2*t_4 - 4*f_4*t_1 + 4*f_4*t_2)) + 8*cos(pi*(-4*f_1*t_2 + 4*f_1*t_4 + 4*f_2*t_1 - 4*f_2*t_4 + 4*f_4*t_1 - 4*f_4*t_2)) + 8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 - 4*f_2*t_1 + 4*f_2*t_3 - 4*f_3*t_1 + 4*f_3*t_2)) + 8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 - 4*f_2*t_1 + 4*f_2*t_3 + 4*f_3*t_1 - 4*f_3*t_2)) + 8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 + 4*f_2*t_1 - 4*f_2*t_3 - 4*f_3*t_1 + 4*f_3*t_2)) + 8*cos(pi*(4*f_1*t_2 - 4*f_1*t_3 + 4*f_2*t_1 - 4*f_2*t_3 + 4*f_3*t_1 - 4*f_3*t_2)) + 8*cos(pi*(-4*f_1*t_3 + 4*f_1*t_4 - 4*f_3*t_1 + 4*f_3*t_4 + 4*f_4*t_1 - 4*f_4*t_3)) + 8*cos(pi*(-4*f_1*t_3 + 4*f_1*t_4 + 4*f_3*t_1 - 4*f_3*t_4 + 4*f_4*t_1 - 4*f_4*t_3)) + 8*cos(pi*(4*f_1*t_3 - 4*f_1*t_4 - 4*f_3*t_1 + 4*f_3*t_4 + 4*f_4*t_1 - 4*f_4*t_3))

<IPython.core.display.Math object>

In [62]:
print("\n copy-pasteable latex string:")
print(to_latex_aligned(moment_4, num_of_deltas_before_linebreak=3))


 copy-pasteable latex string:
\begin{align*}
\mathrm{moment} &= 1.0 + \delta\left(f_{1} + f_{2}\right) \delta\left(t_{1} - t_{2}\right) + \delta\left(f_{1} + f_{3}\right) \delta\left(t_{1} - t_{3}\right) + \delta\left(f_{1} + f_{4}\right) \delta\left(t_{1} - t_{4}\right) \\
&\quad + \delta\left(f_{1} - f_{2}\right) \delta\left(t_{1} - t_{2}\right) + \delta\left(f_{1} - f_{3}\right) \delta\left(t_{1} - t_{3}\right) + \delta\left(f_{1} - f_{4}\right) \delta\left(t_{1} - t_{4}\right) \\
&\quad + \delta\left(f_{2} + f_{3}\right) \delta\left(t_{2} - t_{3}\right) + \delta\left(f_{2} + f_{4}\right) \delta\left(t_{2} - t_{4}\right) + \delta\left(f_{2} - f_{3}\right) \delta\left(t_{2} - t_{3}\right) \\
&\quad + \delta\left(f_{2} - f_{4}\right) \delta\left(t_{2} - t_{4}\right) + \delta\left(f_{3} + f_{4}\right) \delta\left(t_{3} - t_{4}\right) + \delta\left(f_{3} - f_{4}\right) \delta\left(t_{3} - t_{4}\right) \\
&\quad + \delta\left(f_{1} + f_{2}\right) \delta\left(f_{3} + f_{4}\right) \delta\

In [19]:
moment_5 = get_moment_and_simplify(order_of_moment=5, number_of_recursive_simplifications=2, apply_dirac_delta_constraints=True, collect_conjugate_exponential_terms=True, display_intermediate_steps=True)

...Getting correlation structure through Wick's theorem
	Done
...Adding Fourier phase factors to build integrand
	Done
...Performing 5 integrals over 945 terms


Calculated moment 
---------------------
		exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_3*t_3 - f_4*t_3))*DiracDelta(f_1 - f_2)*DiracDelta(f_3 - f_4)*DiracDelta(t_1 - t_2)*DiracDelta(t_3 - t_4) + exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_3*t_3 + f_4*t_3))*DiracDelta(f_1 - f_2)*DiracDelta(f_3 + f_4)*DiracDelta(t_1 - t_2)*DiracDelta(t_3 - t_4) + exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_3*t_5 - f_5*t_5))*DiracDelta(f_1 - f_2)*DiracDelta(f_3 - f_5)*DiracDelta(t_1 - t_2)*DiracDelta(t_3 - t_5) + exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_3*t_5 + f_5*t_5))*DiracDelta(f_1 - f_2)*DiracDelta(f_3 + f_5)*DiracDelta(t_1 - t_2)*DiracDelta(t_3 - t_5) + exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_4*t_5 - f_5*t_5))*DiracDelta(f_1 - f_2)*DiracDelta(f_4 - f_5)*DiracDelta(t_1 - t_2)*DiracDelta(t_4 - t_5) + exp(4*I*pi*(f_1*t_1 - f_2*t_1 + f_4*t_5 + f_5*t_5))*DiracDelta(f_1 - f_2)*DiracDelta(

<IPython.core.display.Math object>



Applying implicit dirac delta constraints 
---------------------
		4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 - f_2*t_2 + f_2*t_3 - f_5*t_1 + f_5*t_2))*DiracDelta(f_1 - f_2 - f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 - f_2*t_2 + f_2*t_3 - f_5*t_1 + f_5*t_2))*DiracDelta(f_1 - f_2 + f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 - f_2*t_2 + f_2*t_3 + f_5*t_1 - f_5*t_2))*DiracDelta(-f_1 + f_2 + f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 - f_2*t_2 + f_2*t_3 + f_5*t_1 - f_5*t_2))*DiracDelta(f_1 - f_2 + f_3 - f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 + f_2*t_2 - f_2*t_3 - f_5*t_1 + f_5*t_2))*DiracDelta(f_1 + f_2 - f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 + f_2*t_2 - f_2*t_3 - f_5*t_1 + f_5*t_2))*DiracDelta(f_1 + f_2 + f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 4*exp(4*I*pi*(-f_1*t_1 + f_1*t_3 + f_2*t_2 - f_2*t_3 + f_5*t

<IPython.core.display.Math object>



Pairing together free conjugate exponentials 
----------------------
		8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 - 4*f_2*t_2 + 4*f_2*t_3 - 4*f_5*t_1 + 4*f_5*t_2))*DiracDelta(f_1 + f_2 - f_3 - f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 - 4*f_2*t_2 + 4*f_2*t_3 - 4*f_5*t_1 + 4*f_5*t_2))*DiracDelta(f_1 + f_2 + f_3 - f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 - 4*f_2*t_2 + 4*f_2*t_3 + 4*f_5*t_1 - 4*f_5*t_2))*DiracDelta(f_1 + f_2 - f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 - 4*f_2*t_2 + 4*f_2*t_3 + 4*f_5*t_1 - 4*f_5*t_2))*DiracDelta(f_1 + f_2 + f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 + 4*f_2*t_2 - 4*f_2*t_3 - 4*f_5*t_1 + 4*f_5*t_2))*DiracDelta(-f_1 + f_2 + f_3 + f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_1*t_1 - 4*f_1*t_3 + 4*f_2*t_2 - 4*f_2*t_3 - 4*f_5*t_1 + 4*f_5*t_2))*DiracDelta(f_1 - f_2 + f_3 - f_5)*DiracDelta(t_1 + t_2 - t_3 - t_5) + 8*cos(pi*(4*f_

<IPython.core.display.Math object>

## Getting the cumulants

In particular, we form:

$$\langle X_1X_2X_3 \rangle^c = \langle X_1X_2X_3 \rangle - \langle X_1X_2 \rangle \langle X_3\rangle-\langle X_2X_3 \rangle \langle X_1\rangle - \langle X_1X_3 \rangle \langle X_2\rangle + 2  \langle X_1\rangle \langle X_2\rangle \langle X_3\rangle$$

In [ ]:
def get_third_cumulant(third_moment):
    pass

In [2]:
import sympy as sym

In [7]:
def S(tuple_1, tuple_2, tuple_3):
    t_1, f_1 = tuple_1
    t_2, f_2 = tuple_2
    t_3, f_3 = tuple_3
    row_1 = sym.cos(4*sym.pi * ( f_1 * t_2 - f_1 * t_3 - f_2 * t_1 + f_2 * t_3 - f_3 * t_1 + f_3 * t_2))
    row_2 = sym.cos(4*sym.pi * ( f_1 * t_2 - f_1 * t_3 - f_2 * t_1 + f_2 * t_3 + f_3 * t_1 - f_3 * t_2))
    row_3 = sym.cos(4*sym.pi * ( f_1 * t_2 - f_1 * t_3 + f_2 * t_1 - f_2 * t_3 - f_3 * t_1 + f_3 * t_2))
    row_4 = sym.cos(4*sym.pi * ( f_1 * t_2 - f_1 * t_3 + f_2 * t_1 - f_2 * t_3 + f_3 * t_1 - f_3 * t_2))
    return sum([row_1, row_2, row_3, row_4])

In [8]:
t1, t2, t3, t4, f1, f2, f3, f4 = sym.symbols('t1, t2, t3, t4, f1, f2, f3, f4')

In [13]:
print(S((t1, f1),(t2, f2),(t3, f3)))

cos(pi*(4*f1*t2 - 4*f1*t3 - 4*f2*t1 + 4*f2*t3 - 4*f3*t1 + 4*f3*t2)) + cos(pi*(4*f1*t2 - 4*f1*t3 - 4*f2*t1 + 4*f2*t3 + 4*f3*t1 - 4*f3*t2)) + cos(pi*(4*f1*t2 - 4*f1*t3 + 4*f2*t1 - 4*f2*t3 - 4*f3*t1 + 4*f3*t2)) + cos(pi*(4*f1*t2 - 4*f1*t3 + 4*f2*t1 - 4*f2*t3 + 4*f3*t1 - 4*f3*t2))
